---
short_title: Daten bereinigen
numbering:
    heading_1: true
    heading_2: true
    title: true
---

# Daten bereinigen

```{caution} Work In Progress
Diese Inhalte sind noch in Bearbeitung.
```

Reale Bibliotheksdaten sind selten sauber. Typische Probleme sind fehlende
Werte, doppelte Datensätze, falsch typisierte Spalten und uneinheitliche
Kategorien. In diesem Kapitel arbeiten wir die Bereinigung Schritt für Schritt
durch – vom Rohimport bis zu einer **tidy** Tabelle.

```{seealso} Vorwissen
:icon: false

Die Grundlagen zu fehlenden Werten und zum Umgang mit Dateien finden Sie in
[Projekt CSV I](../040-Projekt_CSV_I/000-Einleitung.md) und
[Projekt CSV II](../070-Projekt_CSV_II/000-Einleitung.md).
```

In [1]:
import pandas as pd

```{note}
**Hinweis zur Browser-Version:** Der Python-Kernel von JupyterLite (über den
Start-Knopf der Seite) läuft im Browser und hat ein anderes Environment als
Ihre lokale Installation: Die Beispieldatei
`assets/data/bibliothek_unsauber.xlsx` liegt dort nicht im Projekt, und die
Excel-Engine `openpyxl` fehlt. Die folgende Zelle lädt beides **nur im
Browser** von der Kurs-Website nach (erkannt an
`sys.platform == "emscripten"`). Lokal in JupyterLab tut sie nichts.
```

In [ ]:
import sys

# Pyodide (JupyterLite) meldet sich mit der Plattform "emscripten";
# lokal (JupyterLab, Build) ist sie z. B. "darwin" oder "linux".
if sys.platform == "emscripten":
    from pathlib import Path

    from js import location

    # 1) Excel-Engine openpyxl fehlt in JupyterLite. Die Wheels liegen auf
    #    der Kurs-Website (Ordner assets/pypi) und werden direkt installiert.
    import micropip

    _wheels = (
        "et_xmlfile-2.0.0-py3-none-any.whl",
        "openpyxl-3.1.5-py2.py3-none-any.whl",
    )
    _fehler = None
    for _basis in (f"{location.origin}/pypi", "pypi"):
        try:
            await micropip.install(
                [f"{_basis}/{_w}" for _w in _wheels], keep_going=True
            )
            _fehler = None
            break
        except Exception as _e:  # noqa: BLE001
            _fehler = _e
    if _fehler is not None:
        raise RuntimeError(f"openpyxl konnte nicht installiert werden: {_fehler}")

    # 2) Beispieldatei von der Kurs-Website ins virtuelle Dateisystem laden,
    #    sodass relative Pfade wie "../assets/data/..." funktionieren
    def stelle_daten_bereit(pfad):
        """Lädt eine Beispieldatei von der Kurs-Website ins virtuelle Dateisystem."""
        ziel = Path(pfad)
        if ziel.exists():
            return ziel

        from js import XMLHttpRequest

        for url in (f"{location.origin}/{ziel.name}", ziel.name):
            anfrage = XMLHttpRequest.new()
            # synchron möglich, weil der Kernel in einem Web Worker läuft
            anfrage.open("GET", url, False)
            try:
                anfrage.responseType = "arraybuffer"
                anfrage.send()
            except Exception:
                continue  # nächste Kandidaten-URL versuchen
            if anfrage.status == 200:
                inhalt = anfrage.response.to_py()
                if not isinstance(inhalt, str):  # ArrayBuffer -> Bytes
                    ziel.parent.mkdir(parents=True, exist_ok=True)
                    ziel.write_bytes(bytes(inhalt))
                    return ziel
        raise FileNotFoundError(
            f"{pfad} weder lokal noch auf der Kurs-Website gefunden."
        )

    stelle_daten_bereit("../assets/data/bibliothek_unsauber.xlsx")

    # 3) Ausgabeverzeichnis für die Abbildungen sicherstellen
    Path("../assets/090").mkdir(parents=True, exist_ok=True)

In [2]:
rohdaten = pd.read_excel("../assets/data/bibliothek_unsauber.xlsx", sheet_name="Ausleihen")
rohdaten.head()

,Titel,Autor*in,Erscheinungsjahr,Ausleihzahlen,Standort,Medium,Zugangsdatum
0,Einführung in die Bibliothekswissenschaft,NaN,2005,382,zentralbibliothek,buch,22.02.2005
1,Metadaten in der Praxis,"Behrens, Carl",2007,8,Magazin,dvd,2007-03-23
2,Informationskompetenz vermitteln,NaN,2014,3401,zentralbibliothek,DVD,2014-11-20
3,Grundlagen der Katalogisierung,"Dittrich, Erik",unbekannt,186,West,zeitschrift,22.04.2013
4,Digitalisierung im Archiv,"Engel, Friederike",1993,21,MAG,eBook,1993-06-13


## Fehlende Werte erkennen

Mit `isna()` finden Sie fehlende Werte. Die Summe je Spalte zeigt, wie viele
Lücken es gibt.

In [3]:
rohdaten.isna().sum()

Titel               1
Autor*in            7
Erscheinungsjahr    9
Ausleihzahlen       5
Standort            0
Medium              0
Zugangsdatum        6
dtype: int64

## Duplikate entfernen

`duplicated()` markiert Zeilen, die es bereits gibt. `drop_duplicates()`
entfernt sie.

In [4]:
rohdaten.duplicated().sum()

np.int64(5)

In [5]:
rohdaten[rohdaten.duplicated(keep=False)].sort_values(by="Titel").head(10)

,Titel,Autor*in,Erscheinungsjahr,Ausleihzahlen,Standort,Medium,Zugangsdatum
55,Bibliotheken im digitalen Wandel,"Dittrich, Erik",unbekannt,155,Zweigstelle Nord,Zeitschrift,2010-06-12
64,Bibliotheken im digitalen Wandel,"Dittrich, Erik",unbekannt,155,Zweigstelle Nord,Zeitschrift,2010-06-12
28,Digitalisierung im Archiv,"Clausen, Dana",1998,346,Magazin,eBook,18.09.1998
62,Digitalisierung im Archiv,"Clausen, Dana",1998,346,Magazin,eBook,18.09.1998
17,E-Books in Bibliotheken,"Engel, Friederike",2024,318,Zweigstelle Süd,Zeitschr.,NaN
41,E-Books in Bibliotheken,NaN,2021,1100,MAG,Zeitschrift,2021-10-10
61,E-Books in Bibliotheken,"Engel, Friederike",2024,318,Zweigstelle Süd,Zeitschr.,NaN
63,E-Books in Bibliotheken,NaN,2021,1100,MAG,Zeitschrift,2021-10-10
3,Grundlagen der Katalogisierung,"Dittrich, Erik",unbekannt,186,West,zeitschrift,22.04.2013
60,Grundlagen der Katalogisierung,"Dittrich, Erik",unbekannt,186,West,zeitschrift,22.04.2013


In [6]:
df = rohdaten.drop_duplicates().reset_index(drop=True)
df.shape

(60, 7)

## Fehlende Werte behandeln

`dropna()` entfernt jede Zeile mit mindestens einem fehlenden Wert – das kann
sehr viel Datenverlust bedeuten. `fillna()` füllt Lücken gezielt, z. B. mit
einem Ersatztext.

In [7]:
df.dropna().shape

(37, 7)

In [8]:
df.fillna({"Autor*in": "ohne Angabe"}).isna().sum()

Titel               1
Autor*in            0
Erscheinungsjahr    9
Ausleihzahlen       5
Standort            0
Medium              0
Zugangsdatum        5
dtype: int64

**Vorsicht:** Fehlende Werte blind zu füllen verfälscht Auswertungen. Legen Sie
vorher fest, was die Lücke bedeutet (unbekannt, „nicht erfasst“, 0 …). In
unseren Daten lassen wir fehlende Zahlen zunächst stehen und entscheiden erst
nach der Typkonvertierung, wie wir damit umgehen.

## Typen korrigieren: Erscheinungsjahr

Text wie `"unbekannt"` oder Leerzeichen verhindern die Zahlenerkennung.
`.str.strip()` entfernt Leerzeichen, `pd.to_numeric(..., errors="coerce")`
wandelt Unpassendes in einen fehlenden Wert (`NaN`) um.

In [9]:
df["Erscheinungsjahr"] = pd.to_numeric(
    df["Erscheinungsjahr"].astype("string").str.strip(),
    errors="coerce",
)
df["Erscheinungsjahr"].dtype

Int64Dtype()

## Typen korrigieren: Ausleihzahlen

Deutsche Zahlen wie `"8.798"` würden von `to_numeric` fälschlich als `8.798`
gelesen. Eine kleine Funktion entfernt den Tausenderpunkt vor der Konvertierung.

In [10]:
def parse_deutsche_zahl(serie):
    """Wandelt Text mit deutschem Tausenderpunkt in Zahlen um."""
    text = serie.astype("string").str.strip()
    hat_tausenderpunkt = text.str.match(r"^-?\d{1,3}(\.\d{3})+$", na=False)
    text = text.where(~hat_tausenderpunkt, text.str.replace(".", "", regex=False))
    return pd.to_numeric(text, errors="coerce")

In [11]:
df["Ausleihzahlen"] = parse_deutsche_zahl(df["Ausleihzahlen"])
df["Ausleihzahlen"].head(10)

0     382
1       8
2    3401
3     186
4      21
5    <NA>
6     244
7     395
8    8611
9     315
Name: Ausleihzahlen, dtype: Int64

## Typen korrigieren: Zugangsdatum

Datumsangaben liegen in gemischten Formaten vor (`2021-05-03` und
`03.05.2021`). `pd.to_datetime` mit `format="mixed"` und `dayfirst=True` liest
beide Varianten; Unlesbares wird zu `NaT` ("Not a Time").

In [12]:
df["Zugangsdatum"] = pd.to_datetime(
    df["Zugangsdatum"], format="mixed", dayfirst=True, errors="coerce"
)
df["Zugangsdatum"].head()

0   2005-02-22
1   2007-03-23
2   2014-11-20
3   2013-04-22
4   1993-06-13
Name: Zugangsdatum, dtype: datetime64[us]

## Kategorien vereinheitlichen

Über ein Mapping (`dict`) ersetzen wir alle Schreibvarianten durch eine
einheitliche Bezeichnung. `.str.strip()` entfernt dabei störende Leerzeichen.

In [13]:
STANDORT_MAPPING = {
    "ZB": "Zentralbibliothek",
    "zentralbibliothek": "Zentralbibliothek",
    "Nord": "Zweigstelle Nord",
    "Zweigstelle Nord": "Zweigstelle Nord",
    "Süd": "Zweigstelle Süd",
    "Zweigstelle Sued": "Zweigstelle Süd",
    "Zweigstelle Süd": "Zweigstelle Süd",
    "West": "Campus West",
    "campus west": "Campus West",
    "Campus West": "Campus West",
    "mag": "Magazin",
    "MAG": "Magazin",
    "Magazin": "Magazin",
    "Zentralbibliothek": "Zentralbibliothek",
}

In [14]:
MEDIUM_MAPPING = {
    "buch": "Buch",
    "Buch": "Buch",
    "eBook": "E-Book",
    "E-Book": "E-Book",
    "zeitschrift": "Zeitschrift",
    "Zeitschr.": "Zeitschrift",
    "Zeitschrift": "Zeitschrift",
    "dvd": "DVD",
    "DVD": "DVD",
    "hoerbuch": "Hörbuch",
    "Hörbuch": "Hörbuch",
}

In [15]:
df["Standort"] = df["Standort"].str.strip().map(STANDORT_MAPPING)
df["Medium"] = df["Medium"].str.strip().map(MEDIUM_MAPPING)
df["Standort"].value_counts()

Standort
Magazin              13
Zentralbibliothek    12
Zweigstelle Süd      12
Zweigstelle Nord     12
Campus West          11
Name: count, dtype: int64

In [16]:
df["Medium"].value_counts()

Medium
E-Book         15
Buch           12
Zeitschrift    12
DVD            11
Hörbuch        10
Name: count, dtype: int64

## Spalten umbenennen

Sprechende, kleingeschriebene Spaltennamen (snake_case) erleichtern die weitere
Arbeit. `rename(columns={...})` benennt gezielt um.

In [17]:
bibliothek = df.rename(columns={
    "Titel": "titel",
    "Autor*in": "autor",
    "Erscheinungsjahr": "jahr",
    "Ausleihzahlen": "ausleihen",
    "Standort": "standort",
    "Medium": "medium",
    "Zugangsdatum": "zugang",
})
bibliothek.head()

,titel,autor,jahr,ausleihen,standort,medium,zugang
0,Einführung in die Bibliothekswissenschaft,NaN,2005,382,Zentralbibliothek,Buch,2005-02-22
1,Metadaten in der Praxis,"Behrens, Carl",2007,8,Magazin,DVD,2007-03-23
2,Informationskompetenz vermitteln,NaN,2014,3401,Zentralbibliothek,DVD,2014-11-20
3,Grundlagen der Katalogisierung,"Dittrich, Erik",<NA>,186,Campus West,Zeitschrift,2013-04-22
4,Digitalisierung im Archiv,"Engel, Friederike",1993,21,Magazin,E-Book,1993-06-13


## Verbleibende fehlende Werte

Nach der Konvertierung prüfen wir erneut. Zeilen ohne Titel oder Autor\*in
entfernen wir, weil sie sich nicht eindeutig zuordnen lassen. Die fehlenden
Zahlen behalten wir als `NaN` – {term}`Pandas` rechnet damit weiter, ohne sie zu
zählen.

In [18]:
bibliothek.isna().sum()

titel         1
autor         6
jahr         12
ausleihen     6
standort      0
medium        0
zugang        5
dtype: int64

In [19]:
bibliothek = bibliothek.dropna(subset=["titel", "autor"]).reset_index(drop=True)
bibliothek.shape

(53, 7)

## Tidy Data

**Tidy Data** bedeutet: eine Zeile ist eine Beobachtung, eine Spalte ist eine
Variable, und jede Zelle enthält genau einen Wert. Diese Form ist die beste
Ausgangsbasis für Statistik und Visualisierung. Breite Tabellen (ein Medium pro
Spalte) lassen sich mit `melt()` in die lange, tidy Form bringen.

In [20]:
weit = pd.DataFrame({
    "standort": ["Magazin"],
    "Buch": [10],
    "E-Book": [5],
})
weit

,standort,Buch,E-Book
0,Magazin,10,5


In [21]:
lang = weit.melt(id_vars="standort", var_name="medium", value_name="ausleihen")
lang

,standort,medium,ausleihen
0,Magazin,Buch,10
1,Magazin,E-Book,5


## Ergebnis speichern

Die bereinigte Tabelle schreiben wir als neue Excel-Datei weg.

In [22]:
bibliothek.to_excel("../assets/data/bibliothek_sauber.xlsx", index=False)
print("Gespeichert:", len(bibliothek), "saubere Zeilen")

Gespeichert: 53 saubere Zeilen


## Übung

````{exercise}
:label: datenbereinigen-uebung

1. Zählen Sie die fehlenden Werte je Spalte vor und nach der Bereinigung und
   begründen Sie die Unterschiede.
2. Prüfen Sie mit `bibliothek["standort"].isna().sum()`, ob es nicht zuordenbare
   Kategorien gibt.
3. Füllen Sie fehlende `jahr`-Werte mit dem Median und vergleichen Sie das
   Ergebnis mit dem Weglassen der Zeilen. Welche Variante ist hier sinnvoller?

````